# SafeBench AI — HuggingFace Model Runner (Colab)

Run this notebook on Colab to use the free GPU for local HuggingFace models.
At the end, download the results JSON and drop it into your local `results/` folder.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Confirm GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — change runtime to GPU')

In [ ]:
# 2. Upload your project files
from google.colab import files
print('Upload these files from your SafeBenchAI folder:')
print('  core/config.py, core/runner.py')
print('  scoring/scorer.py, scoring/aggregator.py')
print('  run_experiments.py')
print('  data/attack_sequences/sequences.json')
print('  data/benign_sequences/benign.json')
print()
uploaded = files.upload()

In [ ]:
# 3. Reconstruct the project folder structure
import os, shutil

for folder in ['core', 'scoring', 'data/attack_sequences', 'data/benign_sequences', 'results']:
    os.makedirs(folder, exist_ok=True)

moves = {
    'config.py':       'core/config.py',
    'runner.py':       'core/runner.py',
    'scorer.py':       'scoring/scorer.py',
    'aggregator.py':   'scoring/aggregator.py',
    'sequences.json':  'data/attack_sequences/sequences.json',
    'benign.json':     'data/benign_sequences/benign.json',
}
for src, dst in moves.items():
    if os.path.exists(src):
        shutil.move(src, dst)

open('core/__init__.py', 'w').close()
open('scoring/__init__.py', 'w').close()

print('Project structure ready.')
!find . -name '*.py' | grep -v __pycache__ | sort

In [ ]:
# 4. Install dependencies
!pip install -q groq google-generativeai detoxify python-dotenv huggingface_hub transformers accelerate bitsandbytes

In [ ]:
# 5. Set API keys
import os
os.environ['GROQ_API_KEY']   = 'gsk_...'  # fill in — needed for LLM judge signal
os.environ['GEMINI_API_KEY'] = ''          # not needed for HF models
os.environ['RANDOM_SEED']    = '42'

In [ ]:
# 6. Patch runner.py:
#    - 4-bit quantization: Mistral 7B goes from 14GB → ~4GB, fits fully on T4 GPU
#      This is why the previous run took 5 hours — weights were spilling to CPU.
#    - Module-level pipeline cache: model loads once, reused across all 3 defenses

with open('core/runner.py', 'r') as f:
    src = f.read()

# Add module-level cache dict
src = src.replace(
    'logger = logging.getLogger(__name__)\n',
    'logger = logging.getLogger(__name__)\n\n# Module-level cache — survives across ConversationRunner instances (one per defense).\n_hf_pipeline_cache: dict = {}\n'
)

# Remove instance variable
src = src.replace(
    '# Local HF pipeline — loads model on first call, cached for the run\n        self._hf_pipeline  = None',
    '# _hf_pipeline is now a module-level cache; no instance variable needed'
)

# Replace the loading + inference block with 4-bit quantized version
old_method = '''        if self._hf_pipeline is None:
            try:
                import torch
                from transformers import pipeline

                model_id = self.model_cfg["model_id"]
                logger.info(f"Loading local HF model '{model_id}' — first run may download several GB...")

                self._hf_pipeline = pipeline(
                    "text-generation",
                    model=model_id,
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    device_map="auto",
                )
                logger.info("Model loaded.")
            except Exception as e:
                logger.error(f"Failed to load local HF model: {e}")
                return f"[HF_LOAD_ERROR: {e}]"

        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.extend(history)

        try:
            output = self._hf_pipeline(
                messages,
                max_new_tokens=1024,
                do_sample=False,
                temperature=None,
                pad_token_id=self._hf_pipeline.tokenizer.eos_token_id,
            )
            # pipeline returns the full conversation; the last entry is the new reply
            return output[0]["generated_text"][-1]["content"].strip()'''

new_method = '''        model_id = self.model_cfg["model_id"]
        if model_id not in _hf_pipeline_cache:
            try:
                import torch
                from transformers import pipeline, BitsAndBytesConfig

                logger.info(f"Loading '{model_id}' in 4-bit quantization (fits T4 GPU)...")
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                )
                _hf_pipeline_cache[model_id] = pipeline(
                    "text-generation",
                    model=model_id,
                    model_kwargs={"quantization_config": bnb_config},
                    device_map="auto",
                )
                logger.info("Model loaded.")
            except Exception as e:
                logger.error(f"Failed to load local HF model: {e}")
                return f"[HF_LOAD_ERROR: {e}]"

        pipe = _hf_pipeline_cache[model_id]
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.extend(history)

        try:
            output = pipe(
                messages,
                max_new_tokens=1024,
                pad_token_id=pipe.tokenizer.eos_token_id,
            )
            return output[0]["generated_text"][-1]["content"].strip()'''

src = src.replace(old_method, new_method)

with open('core/runner.py', 'w') as f:
    f.write(src)

print('runner.py patched — 4-bit quantization + module-level cache.')

In [ ]:
# 7. Patch scorer.py — skip LLM judge instantly when GROQ_API_KEY is empty
# Without this, every turn waits ~6s retrying a call that will always fail.
with open('scoring/scorer.py', 'r') as f:
    src = f.read()

src = src.replace(
    '''    if harm_category == "benign" or not harm_category:
        return {"signal": "llm_judge", "fired": False, "note": "skipped_for_benign"}''',
    '''    if harm_category == "benign" or not harm_category:
        return {"signal": "llm_judge", "fired": False, "note": "skipped_for_benign"}

    from core.config import GROQ_API_KEY
    if not GROQ_API_KEY:
        return {"signal": "llm_judge", "fired": False, "note": "skipped_no_groq_key"}'''
)

with open('scoring/scorer.py', 'w') as f:
    f.write(src)

print('scorer.py patched.')

In [ ]:
# 8a. Run Mistral 7B — full ablation (all 3 defenses, all sequences)
!python run_experiments.py \
    --models hf/mistral-7b-instruct \
    --defenses none system_prompt system_prompt_plus_classifier

In [ ]:
# 8b. Run Phi-3 Mini — full ablation (all 3 defenses, all sequences)
!python run_experiments.py \
    --models hf/phi-3-mini-4k-instruct \
    --defenses none system_prompt system_prompt_plus_classifier

In [ ]:
# 9. (Optional) Re-score with LLM judge if GROQ_API_KEY was set in cell 5
import json, glob
from scoring.scorer import score_result

for results_file in sorted(glob.glob('results/run_*.json')):
    if '_metrics' in results_file:
        continue
    print(f'Re-scoring {results_file} ...')
    with open(results_file) as f:
        data = json.load(f)
    for result in data.get('results', []):
        score_result(result)
    with open(results_file, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  done.')
print('All files re-scored.')

In [ ]:
# 10. Download results
import glob
from google.colab import files

for f in sorted(glob.glob('results/run_*.json')):
    if '_metrics' not in f:
        print(f'Downloading {f} ...')
        files.download(f)

## Back on your local machine

Once the JSONs are in your local `results/` folder, run the aggregator on each:

```bash
python -m scoring.aggregator results/run_YYYYMMDD_HHMMSS.json
```